# EEG Seizure Detection Classifier

This project uses one second EEG segments to classify seizure versus non seizure activity.
'y' has 5 values. Class 1 = seizure. Classes 2–5 are all "not seizure" for different reasons (tumor region, healthy region, eyes open, eyes closed).

## Goals
- Build an interpretable logistic regression baseline
- Compare it with a random forest
- Evaluate seizure detection using precision, recall, and F1 score
- Discuss limitations and potential neurotechnology applications

In [ ]:
import sys 
from pathlib import Path 

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
FIGURES_DIR = PROJECT_DIR / "figures"

DATA_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

print("Python environment:", sys.executable)
print("Project directory:", PROJECT_DIR)
print("Data directory created:", DATA_DIR.exists())
print("Figures directory created:", FIGURES_DIR.exists())

In [ ]:
DATA_PATH = DATA_DIR / "epileptic_seizure_data.csv"

print("Dataset found:", DATA_PATH.exists())


if DATA_PATH.exists():
    size_mb = DATA_PATH.stat().st_size / (1024**2)
    print(f"File size: {size_mb:.2f} MB")

import pandas as pd

df = pd.read_csv(DATA_DIR / "epileptic_seizure_data.csv", index_col=0)

print("Shape:", df.shape)
df.head()

In [ ]:
df["y"].value_counts().sort_index()

In [ ]:
df["target"] = (df["y"] == 1).astype(int) #df["y"] == 1 creates a column of True/False values, astype(int) converts it into binary 0 or 1

print(df["target"].value_counts())
print("Seizure proportion: {:.2%}".format(df["target"].mean()))

# Training and standardisation
We require a stratified testing split as the data is split between 20% epileptic seizure and 80% non epileptic episodes. we then have to use logistic regression to standardise the data, so we rescale every column to have mean 0 and standard deviation 1.

In [ ]:
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler

feature_cols = [c for c in df.columns if c.startswith("X")] #grabs just the 178 X columns, leaving out y and target
X = df[feature_cols]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size = 0.2, 
    stratify = y, 
    random_state = 42 #makes the split reproducible
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train shape:", X_train_scaled.shape)
print("Test shape:", X_test_scaled.shape)
print("Train seizure rate: {:.2%}".format(y_train.mean()))
print("Test seizure rate: {:.2%}".format(y_test.mean()))


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

log_reg = LogisticRegression(
    max_iter=2000,
    C=0.1,
    solver="lbfgs",
    random_state=42
)
log_reg.fit(X_train_scaled, y_train)

print("Logistic regression trained.")


In [ ]:
import numpy as np

print("Max abs value in X_train_scaled:", np.abs(X_train_scaled).max())
print("Max abs value in raw X_train:", X_train.abs().max().max())

# which column and row has it
col_max = X_train.abs().max().idxmax()
print("Worst column:", col_max, "max abs raw value:", X_train[col_max].abs().max())

In [ ]:
from sklearn.metrics import accuracy_score

train_acc = log_reg.score(X_train_scaled, y_train)
test_acc = log_reg.score(X_test_scaled, y_test)

print("Train accuracy:", train_acc)
print("Test accuracy:", test_acc)
print("Any NaN in coefficients?", np.isnan(log_reg.coef_).any())

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)  # unscaled — trees don't need scaling

print("Random forest trained.")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

models = {
    "Logistic Regression": (log_reg, X_test_scaled),
    "Random Forest": (rf, X_test)
}

for name, (model, X_eval) in models.items():
    preds = model.predict(X_eval)
    print(f"--- {name} ---")
    print(classification_report(y_test, preds, target_names=["Non-seizure", "Seizure"]))

    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Non-seizure", "Seizure"])
    disp.plot(cmap="Blues")
    plt.title(f"Confusion Matrix — {name}")
    plt.savefig(FIGURES_DIR / f"confusion_matrix_{name.replace(' ', '_').lower()}.png", bbox_inches="tight")
    plt.show()

In [ ]:
log_reg_balanced = LogisticRegression(
    max_iter=2000,
    C=0.1,
    solver="lbfgs",
    class_weight="balanced",
    random_state=42
)
log_reg_balanced.fit(X_train_scaled, y_train)

rf_balanced = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42
)
rf_balanced.fit(X_train, y_train)

for name, (model, X_eval) in {
    "Logistic Regression (balanced)": (log_reg_balanced, X_test_scaled),
    "Random Forest (balanced)": (rf_balanced, X_test)
}.items():
    preds = model.predict(X_eval)
    print(f"--- {name} ---")
    print(classification_report(y_test, preds, target_names=["Non-seizure", "Seizure"]))

    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Non-seizure", "Seizure"])
    disp.plot(cmap="Blues")
    plt.title(f"Confusion Matrix — {name}")
    plt.savefig(FIGURES_DIR / f"confusion_matrix_{name.replace(' ', '_').lower()}.png", bbox_inches="tight")
    plt.show()

In [ ]:
from sklearn.metrics import precision_recall_curve

probs = rf.predict_proba(X_test)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_test, probs)

plt.plot(thresholds, precisions[:-1], label="Precision")
plt.plot(thresholds, recalls[:-1], label="Recall")
plt.xlabel("Decision threshold")
plt.legend()
plt.title("Precision/Recall vs. threshold — Random Forest")
plt.savefig(FIGURES_DIR / "threshold_tuning_rf.png", bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

for t in [0.3, 0.4, 0.5]:
    preds_t = (probs >= t).astype(int)
    p = precision_score(y_test, preds_t)
    r = recall_score(y_test, preds_t)
    f1 = f1_score(y_test, preds_t)
    print(f"Threshold {t}: precision={p:.3f}, recall={r:.3f}, f1={f1:.3f}")

In [ ]:
importances = rf.feature_importances_

plt.figure(figsize=(10, 4))
plt.plot(range(1, 179), importances)
plt.xlabel("Time point (X1–X178, across the 1-second window)")
plt.ylabel("Feature importance")
plt.title("Random Forest Feature Importance Across the EEG Window")
plt.savefig(FIGURES_DIR / "feature_importance_rf.png", bbox_inches="tight")
plt.show()

# Also print the single most important time point
top_idx = importances.argmax() + 1  # +1 because X1 corresponds to index 0
print(f"Most important feature: X{top_idx} (importance={importances.max():.4f})")